# Data Exploration and Databricks Integration

This notebook demonstrates:
1. Connecting to Databricks (simulated)
2. Loading training data
3. Data quality analysis
4. Preprocessing for LLM fine-tuning

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from data.data_loader import DatabricksConnector, DataPreprocessor

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Connect to Data Source

In production, this would connect to a real Databricks workspace.
For this demo, we're using simulated data.

In [ ]:
# Initialize Databricks connector in simulation mode
connector = DatabricksConnector(
    use_simulation=True,
    catalog="portfolio_demo",
    schema="ml_training"
)

# Show table schema
schema = connector.get_table_schema("customer_support_qa")
print("\nTable Schema:")
for col in schema["columns"]:
    print(f"  {col['name']:20s} ({col['type']:10s}) - {col['description']}")

## 2. Load Training Data

In [ ]:
# Load dataset
dataset = connector.load_training_data(limit=1000)

print(f"Dataset size: {len(dataset)} examples")
print(f"Features: {dataset.column_names}")

# Convert to pandas for analysis
df = pd.DataFrame(dataset)

In [ ]:
# Display sample data
df.head(10)

## 3. Data Quality Analysis

In [ ]:
# Text length statistics
df['instruction_length'] = df['instruction'].str.split().str.len()
df['response_length'] = df['response'].str.split().str.len()

print("Text Length Statistics:")
print("\nInstruction length:")
print(df['instruction_length'].describe())
print("\nResponse length:")
print(df['response_length'].describe())

In [ ]:
# Plot length distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['instruction_length'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_title('Instruction Length Distribution')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['response_length'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title('Response Length Distribution')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Category distribution
category_counts = df['category'].value_counts()

plt.figure(figsize=(10, 6))
category_counts.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Support Category Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nCategory Distribution:")
print(category_counts)

In [ ]:
# Quality score distribution
if 'quality_score' in df.columns:
    plt.figure(figsize=(10, 5))
    plt.hist(df['quality_score'], bins=20, edgecolor='black', alpha=0.7, color='green')
    plt.title('Quality Score Distribution', fontsize=14, fontweight='bold')
    plt.xlabel('Quality Score')
    plt.ylabel('Frequency')
    plt.axvline(df['quality_score'].mean(), color='red', linestyle='--', label=f'Mean: {df["quality_score"].mean():.2f}')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"\nQuality Score Statistics:")
    print(df['quality_score'].describe())

## 4. Data Preprocessing

Format data in Alpaca instruction format for fine-tuning

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor(format="alpaca")

# Prepare dataset
prepared_data = preprocessor.prepare_dataset(dataset, train_split=0.9)

print(f"Training examples: {len(prepared_data['train'])}")
print(f"Validation examples: {len(prepared_data['validation'])}")

In [ ]:
# Show formatted example
print("Formatted Training Example:")
print("=" * 80)
print(prepared_data['train'][0]['text'])
print("=" * 80)

## 5. Data Quality Checks

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Check for duplicates
duplicates = df.duplicated(subset=['instruction']).sum()
print(f"\nDuplicate instructions: {duplicates}")

# Data quality summary
print("\n" + "="*50)
print("DATA QUALITY SUMMARY")
print("="*50)
print(f"Total examples: {len(df)}")
print(f"Unique categories: {df['category'].nunique()}")
print(f"Avg instruction length: {df['instruction_length'].mean():.1f} words")
print(f"Avg response length: {df['response_length'].mean():.1f} words")
if 'quality_score' in df.columns:
    print(f"Avg quality score: {df['quality_score'].mean():.2f}")
    print(f"High quality (>0.9): {(df['quality_score'] > 0.9).sum()} ({(df['quality_score'] > 0.9).sum()/len(df)*100:.1f}%)")

## 6. Save Processed Dataset

In [ ]:
# Save dataset for training
connector.save_dataset_locally(dataset, "sample_customer_support.json")

print("Dataset saved to data/raw/sample_customer_support.json")
print("\nReady for training!")

## Summary

This notebook demonstrated:
- ✅ Databricks connector pattern (simulated)
- ✅ Data loading and exploration
- ✅ Quality analysis and visualization
- ✅ Data preprocessing for LLM fine-tuning
- ✅ Alpaca instruction formatting

**Next Steps:**
1. Proceed to `02_training_demo.ipynb` for model fine-tuning
2. Or run the training script: `python training/train_sft.py`